In [1]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [42]:
df= pd.read_csv("C:\\Users\\sarah\\Desktop\\Groceries_dataset.csv")
df.head(5)

,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


## data understanding and cleaning 

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38765 entries, 0 to 38764
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Member_number    38765 non-null  int64 
 1   Date             38765 non-null  object
 2   itemDescription  38765 non-null  object
dtypes: int64(1), object(2)
memory usage: 908.7+ KB


In [44]:
df['itemDescription'] = df['itemDescription'].str.strip()
df.Date = pd.to_datetime(data.Date)
df["Member_number"] = data['Member_number'].astype("string")

In [45]:
df.dtypes

Member_number              string
Date               datetime64[ns]
itemDescription            object
dtype: object

In [46]:
df.shape

(38765, 3)

## performing MBA 

In [47]:
basket = df.groupby(['Member_number', 'itemDescription'])['itemDescription'].count().unstack().fillna(0).reset_index()
basket.head()

itemDescription,Member_number,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,1000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0
1,1001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0
2,1002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,1003,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0


In [48]:
def encoder(k):
    if k <= 0:
        return 0
    if k >= 1:
        return 1

In [49]:
basket =basket.iloc[:, 1:basket.shape[1]].applymap(encoder)
basket

itemDescription,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0
1,0,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,1,0,1,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3893,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3894,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,0,0
3895,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3896,0,0,0,0,0,0,0,0,0,1,...,0,0,0,1,0,0,0,0,1,0


In [53]:
frequent_itemsets = apriori(basket, min_support=0.07, use_colnames=True).sort_values(by='support')
frequent_itemsets

,support,itemsets
47,0.070292,"(whole milk, domestic eggs)"
69,0.071575,"(yogurt, root vegetables)"
49,0.071575,"(other vegetables, pastry)"
82,0.071832,"(other vegetables, yogurt, whole milk)"
45,0.071832,"(citrus fruit, rolls/buns)"
...,...,...
37,0.282966,(yogurt)
32,0.313494,(soda)
28,0.349666,(rolls/buns)
24,0.376603,(other vegetables)


## performing confidence-support-lift

In [55]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1).sort_values('lift', ascending=False)
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
10,(yogurt),"(other vegetables, whole milk)",0.282966,0.191380,0.071832,0.253853,1.326434,0.017678,1.083727
7,"(other vegetables, whole milk)",(yogurt),0.191380,0.282966,0.071832,0.375335,1.326434,0.017678,1.147870
11,(whole milk),"(other vegetables, yogurt)",0.458184,0.120318,0.071832,0.156775,1.303003,0.016704,1.043235
6,"(other vegetables, yogurt)",(whole milk),0.120318,0.458184,0.071832,0.597015,1.303003,0.016704,1.344507
21,(sausage),(yogurt),0.206003,0.282966,0.075423,0.366127,1.293892,0.017132,1.131196


## performing at least 3 filters

In [57]:
basket['whole milk'].sum()

1786

In [61]:
basket['sausage'].sum()

803

In [60]:
rules[ (rules['lift'] >= 1.2) & (rules['confidence'] >= 0.3) ]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
7,"(other vegetables, whole milk)",(yogurt),0.191380,0.282966,0.071832,0.375335,1.326434,0.017678,1.147870
6,"(other vegetables, yogurt)",(whole milk),0.120318,0.458184,0.071832,0.597015,1.303003,0.016704,1.344507
21,(sausage),(yogurt),0.206003,0.282966,0.075423,0.366127,1.293892,0.017132,1.131196
8,"(yogurt, whole milk)",(other vegetables),0.150590,0.376603,0.071832,0.477002,1.266589,0.015119,1.191967
38,"(other vegetables, whole milk)",(rolls/buns),0.191380,0.349666,0.082093,0.428954,1.226753,0.015174,1.138847
39,"(other vegetables, rolls/buns)",(whole milk),0.146742,0.458184,0.082093,0.559441,1.220996,0.014859,1.229837
40,"(whole milk, rolls/buns)",(other vegetables),0.178553,0.376603,0.082093,0.459770,1.220834,0.014850,1.153947


In [62]:
rules[(rules['confidence'] <= 0.5)]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,leverage,conviction
10,(yogurt),"(other vegetables, whole milk)",0.282966,0.191380,0.071832,0.253853,1.326434,0.017678,1.083727
7,"(other vegetables, whole milk)",(yogurt),0.191380,0.282966,0.071832,0.375335,1.326434,0.017678,1.147870
11,(whole milk),"(other vegetables, yogurt)",0.458184,0.120318,0.071832,0.156775,1.303003,0.016704,1.043235
21,(sausage),(yogurt),0.206003,0.282966,0.075423,0.366127,1.293892,0.017132,1.131196
20,(yogurt),(sausage),0.282966,0.206003,0.075423,0.266546,1.293892,0.017132,1.082545
...,...,...,...,...,...,...,...,...,...
93,(whole milk),(soda),0.458184,0.313494,0.151103,0.329787,1.051973,0.007465,1.024310
86,(other vegetables),(soda),0.376603,0.313494,0.124166,0.329700,1.051695,0.006103,1.024178
87,(soda),(other vegetables),0.313494,0.376603,0.124166,0.396072,1.051695,0.006103,1.032237
58,(other vegetables),(tropical fruit),0.376603,0.233710,0.091329,0.242507,1.037642,0.003313,1.011614


# interpretation
## here we see items with strong relationships for example (yogurt) has a strong relationship with (other vegetables and whole milk ) now we can come up with action plans such as making the yogurt section near the vegetables and whole milk sections so it becomes easier for customers to buy these things together and the market makes more profit 